# Module 01 — Reduced Steady State (RSS) Finder

**Repository:** `quantum-relaxation-ordering`  
**Stewards:** Colla, A. · Warring, U.  
**Affiliation:** Theory · Numerical · Experimental Quantum & Atomic Physics, Freiburg  
**Version:** 0.1  

---

## Purpose

This notebook computes the **reduced steady state (RSS)** of the spin subsystem in a spin-motion-bath model. It:

1. Defines the spin + single motional mode system with Markovian bath
2. Solves for the Liouvillian steady state
3. Traces out the motional mode to obtain ρ_ss^{spin}
4. Runs the **RSS diagnostic check**: initialise spin in ρ_ss^{spin}, motion thermal → confirm D(ρ_spin(t), ρ_ss) rises then returns
5. Performs a **parameter scan** over (g, n̄, γ, η)
6. Exports results to `../data/rss_dynamics.json` for website rendering

## Pre-registration note

This notebook produces **illustrative simulations only**. Parameters are chosen for conceptual clarity, not platform calibration. Platform-calibrated runs will be added in a later module after pre-registration of the analysis protocol.

## Dependencies

```bash
pip install qutip numpy matplotlib scipy tqdm
```

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import json
import os
from datetime import datetime

import qutip as qt
from qutip import mesolve, steadystate, tensor, qeye, destroy
from qutip import sigmaz, sigmax, sigmam, thermal_dm, ket2dm, basis

print(f"QuTiP version : {qt.__version__}")
print(f"NumPy version : {np.__version__}")
print(f"Run timestamp : {datetime.now().isoformat()}")

In [ ]:
# ── Plot style ─────────────────────────────────────────────────────────────
NAVY   = '#0d1b2a'
AMBER  = '#f0c46a'
BLUE   = '#5b9bd5'
CREAM  = '#f5f0e8'
GREY   = '#8aaabb'

mpl.rcParams.update({
    'figure.facecolor' : NAVY,
    'axes.facecolor'   : NAVY,
    'axes.edgecolor'   : '#2a4060',
    'axes.labelcolor'  : GREY,
    'xtick.color'      : GREY,
    'ytick.color'      : GREY,
    'text.color'       : CREAM,
    'grid.color'       : '#1e3050',
    'grid.linestyle'   : '--',
    'font.family'      : 'monospace',
    'figure.dpi'       : 120,
})

---
## 1. System Definition

**Hamiltonian:**

$$H = \frac{\omega_0}{2}\sigma_z + \omega_m a^\dagger a + g\,(\sigma_+ + \sigma_-)(a + a^\dagger)$$

**Markovian bath** (motional mode coupled to thermal reservoir):

$$\mathcal{L}\rho = \gamma(\bar{n}+1)\left(a\rho a^\dagger - \tfrac{1}{2}\{a^\dagger a, \rho\}\right) + \gamma\bar{n}\left(a^\dagger\rho a - \tfrac{1}{2}\{a a^\dagger, \rho\}\right)$$

**Lamb-Dicke correction:** replaces $g(a+a^\dagger)$ coupling with the full displacement operator expansion to order $\eta^2$.

In [ ]:
def build_system(omega0=1.0, omegam=1.0, g=0.05, gamma=0.02,
                 n_bar=5.0, N=25, lamb_dicke_eta=0.0):
    """
    Build the spin + motional mode Hamiltonian and jump operators.

    Parameters
    ----------
    omega0         : float  — spin transition frequency (units: ω₀ = 1)
    omegam         : float  — motional mode frequency
    g              : float  — spin-motion coupling (Lamb-Dicke regime: g ≪ ωm)
    gamma          : float  — Markovian bath decay rate
    n_bar          : float  — mean phonon number of bath
    N              : int    — Fock space truncation
    lamb_dicke_eta : float  — Lamb-Dicke parameter η (0 = pure LD approximation)

    Returns
    -------
    H      : Qobj  — total Hamiltonian
    c_ops  : list  — collapse operators
    ops    : dict  — named operators for later use
    """
    # Basis operators
    sm  = tensor(sigmam(), qeye(N))           # spin lowering
    sp  = sm.dag()                             # spin raising
    sz  = tensor(sigmaz(), qeye(N))            # spin z
    a   = tensor(qeye(2),  destroy(N))         # motional lowering
    ada = a.dag() * a                          # phonon number

    # Coupling: Lamb-Dicke limit (η=0) or first correction
    if lamb_dicke_eta == 0.0:
        # Pure Lamb-Dicke: linear coupling
        x_op = a + a.dag()
    else:
        # First LD correction: expand e^{iη(a+a†)} to order η²
        # e^{iηx} ≈ 1 + iηx - η²x²/2
        x_op = (a + a.dag())
        x2   = x_op * x_op
        # Effective coupling operator (keeping η² term)
        eff  = x_op - (lamb_dicke_eta**2 / 2.0) * x2
        x_op = eff

    # Hamiltonian
    H = (omega0 / 2.0) * sz \
      + omegam * ada \
      + g * (sp + sm) * x_op

    # Jump operators (motional mode ↔ thermal bath)
    c_ops = [
        np.sqrt(gamma * (n_bar + 1.0)) * a,        # phonon decay
        np.sqrt(gamma * n_bar)         * a.dag(),  # phonon creation
    ]

    ops = dict(sm=sm, sp=sp, sz=sz, a=a, ada=ada)
    return H, c_ops, ops

---
## 2. Reduced Steady State Solver

In [ ]:
def get_rss(H, c_ops, spin_index=0, method='direct'):
    """
    Compute reduced steady state of the spin subsystem.

    Parameters
    ----------
    H          : Qobj  — Hamiltonian
    c_ops      : list  — collapse operators
    spin_index : int   — index of spin in tensor product (0 = first)
    method     : str   — QuTiP steadystate solver method

    Returns
    -------
    rho_ss_total : Qobj  — full steady state density matrix
    rho_ss_spin  : Qobj  — reduced steady state (spin only)
    bloch_vec    : array — [x, y, z] Bloch vector of rho_ss_spin
    """
    rho_ss_total = steadystate(H, c_ops, method=method)
    rho_ss_spin  = rho_ss_total.ptrace(spin_index)

    # Bloch vector
    from qutip import sigmax, sigmay
    sx = tensor(sigmax(), qeye(rho_ss_total.dims[0][1]))
    sy = tensor(qt.sigmay(), qeye(rho_ss_total.dims[0][1]))
    sz = tensor(sigmaz(), qeye(rho_ss_total.dims[0][1]))

    bx = float(qt.expect(sx, rho_ss_total))
    by = float(qt.expect(sy, rho_ss_total))
    bz = float(qt.expect(sz, rho_ss_total))

    return rho_ss_total, rho_ss_spin, np.array([bx, by, bz])


def trace_distance(rho, sigma):
    """
    Trace distance D(rho, sigma) = ½ Tr|rho - sigma|.
    For qubit states uses Bloch vector formula for speed.
    """
    diff = rho - sigma
    # Eigenvalues of Hermitian diff
    evals = diff.eigenenergies()
    return 0.5 * float(np.sum(np.abs(evals)))

In [ ]:
# ── Baseline parameters ─────────────────────────────────────────────────────
PARAMS_BASE = dict(
    omega0         = 1.0,
    omegam         = 1.0,
    g              = 0.05,
    gamma          = 0.02,
    n_bar          = 5.0,
    N              = 20,
    lamb_dicke_eta = 0.0,
)

H, c_ops, ops = build_system(**PARAMS_BASE)
rho_ss_total, rho_ss_spin, bloch_vec = get_rss(H, c_ops)

print("Reduced Steady State (spin):")
print(rho_ss_spin)
print(f"\nBloch vector: [{bloch_vec[0]:.4f}, {bloch_vec[1]:.4f}, {bloch_vec[2]:.4f}]")
print(f"Purity: {float((rho_ss_spin * rho_ss_spin).tr()):.4f}")

---
## 3. Liouvillian Spectrum

In [ ]:
def liouvillian_spectrum(H, c_ops, n_evals=8):
    """
    Compute the n_evals eigenvalues of the Liouvillian superoperator
    with largest real part (closest to zero).
    Returns eigenvalues sorted by |Re(λ)|.
    """
    L = qt.liouvillian(H, c_ops)
    # Convert to dense for eigs — sparse for large systems
    evals = L.eigenenergies(sparse=True, sort='high', eigvals=n_evals)
    return np.sort(evals, key=lambda x: abs(x.real))


evals = liouvillian_spectrum(H, c_ops, n_evals=6)

print("Liouvillian eigenvalues (sorted by |Re(λ)|):")
print(f"{'k':>3}  {'Re(λ)':>12}  {'Im(λ)':>12}  {'τ_k = -1/Re(λ)':>18}")
print("-" * 55)
for k, ev in enumerate(evals):
    re = ev.real
    im = ev.imag
    tau = -1.0/re if re < -1e-12 else float('inf')
    print(f"{k:>3}  {re:>12.6f}  {im:>12.6f}  {tau:>18.4f}")

print(f"\nRelaxation gap Δ = |Re(λ₁)| = {abs(evals[1].real):.6f}")
print(f"Dominant relaxation time τ   = {-1/evals[1].real:.4f}  (units: 1/ω₀)")

---
## 4. RSS Diagnostic Check

Initialise spin in ρ_ss^{spin}, motion in thermal state at n̄.  
D(ρ_spin(t), ρ_ss) should **rise then return to zero**.  
This confirms ρ_ss^{spin} is a global but not local fixed point of spin dynamics.

In [ ]:
def rss_diagnostic(H, c_ops, rho_ss_spin, rho_ss_total,
                   n_bar_init=5.0, N=20, t_end=200, n_steps=80):
    """
    RSS diagnostic check: spin in RSS, motion thermal.
    Returns time array and D(rho_spin(t), rho_ss) array.
    """
    # Initial state: spin in RSS ⊗ motion thermal
    rho_motion_init = thermal_dm(N, n_bar_init)
    rho0 = tensor(rho_ss_spin, rho_motion_init)

    times = np.linspace(0, t_end, n_steps)
    result = mesolve(H, rho0, times, c_ops, [])

    D_vals = np.array([
        trace_distance(state.ptrace(0), rho_ss_spin)
        for state in result.states
    ])
    return times, D_vals


print("Running RSS diagnostic check...")
times_diag, D_diag = rss_diagnostic(
    H, c_ops, rho_ss_spin, rho_ss_total,
    n_bar_init=PARAMS_BASE['n_bar'],
    N=PARAMS_BASE['N'],
    t_end=200, n_steps=60
)
print(f"Peak D  = {D_diag.max():.4f}  at t = {times_diag[D_diag.argmax()]:.1f}")
print(f"Final D = {D_diag[-1]:.6f}  (should approach zero)")

# Plot
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(times_diag, D_diag, color=AMBER, lw=2)
ax.fill_between(times_diag, D_diag, alpha=0.08, color=AMBER)
ax.axhline(0, color='white', lw=0.5, alpha=0.3)
ax.set_xlabel('time  (units: 1/ω₀)')
ax.set_ylabel('D(ρ_spin(t), ρ_ss)')
ax.set_title('RSS Diagnostic Check — spin initialised in RSS, motion thermal at n̄=5',
             fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. Mpemba Scenario (Illustrative State Pair)

Construct two initial states:
- **State A**: farther from ρ_ss (excited spin, D₀ large)
- **State B**: closer to ρ_ss (partial mixture, D₀ moderate)

Test whether A crosses below B in trace distance.

In [ ]:
def evolve_spin_distance(H, c_ops, rho_spin_init, rho_ss_spin,
                         rho_motion_init, times):
    """
    Evolve from rho_spin_init ⊗ rho_motion_init.
    Return D(rho_spin(t), rho_ss_spin) trajectory.
    """
    rho0 = tensor(rho_spin_init, rho_motion_init)
    result = mesolve(H, rho0, times, c_ops, [])
    return np.array([
        trace_distance(state.ptrace(0), rho_ss_spin)
        for state in result.states
    ])


N = PARAMS_BASE['N']
rho_motion = thermal_dm(N, PARAMS_BASE['n_bar'])
times_mp   = np.linspace(0, 200, 80)

# State A: excited spin |↑⟩⟨↑| — far from RSS
rho_A = ket2dm(basis(2, 0))

# State B: mixture closer to RSS
# Use a point partway between RSS and excited state
alpha = 0.5  # mixing parameter; tune to get D_B < D_A
rho_B = alpha * rho_ss_spin + (1 - alpha) * ket2dm(basis(2, 0))
rho_B = rho_B / rho_B.tr()  # normalise

D0_A = trace_distance(rho_A, rho_ss_spin)
D0_B = trace_distance(rho_B, rho_ss_spin)
print(f"D(ρ_A(0), ρ_ss) = {D0_A:.4f}   [farther]")
print(f"D(ρ_B(0), ρ_ss) = {D0_B:.4f}   [closer]")
assert D0_A > D0_B, "State A must start farther than B"

print("\nEvolving state A...")
D_A = evolve_spin_distance(H, c_ops, rho_A, rho_ss_spin, rho_motion, times_mp)
print("Evolving state B...")
D_B = evolve_spin_distance(H, c_ops, rho_B, rho_ss_spin, rho_motion, times_mp)

# Find crossing
crossing_idx = np.where(D_A < D_B)[0]
if len(crossing_idx) > 0:
    t_cross = times_mp[crossing_idx[0]]
    print(f"\n✓ Crossing found at t* ≈ {t_cross:.2f}  (Mpemba scenario confirmed)")
else:
    print("\n✗ No crossing found — adjust state pair or parameters")
    t_cross = None

In [ ]:
# ── Plot Mpemba scenario ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(times_mp, D_A, color=AMBER, lw=2.5, label=f'State A  (D₀ = {D0_A:.3f}, farther)')
ax.plot(times_mp, D_B, color=BLUE,  lw=2.5, label=f'State B  (D₀ = {D0_B:.3f}, closer)')

if t_cross is not None:
    ax.axvline(t_cross, color='white', lw=1, ls='--', alpha=0.4)
    ax.text(t_cross + 2, 0.02, f't* ≈ {t_cross:.1f}', color='white',
            fontsize=9, alpha=0.6)

ax.set_xlabel('time  (units: 1/ω₀)')
ax.set_ylabel('D(ρ_spin(t), ρ_ss)')
ax.set_title('Mpemba Scenario — illustrative state pair', fontsize=10)
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Parameter Scan: Lamb-Dicke Correction

Scan over η = 0, 0.05, 0.10, 0.15, 0.20.  
For each: compute RSS, crossing time, and D trajectory for the same state pair.

In [ ]:
eta_values = [0.0, 0.05, 0.10, 0.15, 0.20]
scan_results = []

fig, axes = plt.subplots(1, len(eta_values), figsize=(14, 3.5), sharey=True)
fig.suptitle('Lamb-Dicke correction scan — D(ρ_spin(t), ρ_ss)', fontsize=10)

for i, eta in enumerate(eta_values):
    params = dict(**PARAMS_BASE, lamb_dicke_eta=eta)
    H_i, c_ops_i, _ = build_system(**params)
    _, rho_ss_i, _ = get_rss(H_i, c_ops_i)

    D0_A_i = trace_distance(rho_A, rho_ss_i)
    D0_B_i = trace_distance(rho_B, rho_ss_i)

    rho_motion_i = thermal_dm(PARAMS_BASE['N'], PARAMS_BASE['n_bar'])
    D_A_i = evolve_spin_distance(H_i, c_ops_i, rho_A, rho_ss_i, rho_motion_i, times_mp)
    D_B_i = evolve_spin_distance(H_i, c_ops_i, rho_B, rho_ss_i, rho_motion_i, times_mp)

    cross_idx_i = np.where(D_A_i < D_B_i)[0]
    t_cross_i = float(times_mp[cross_idx_i[0]]) if len(cross_idx_i) > 0 else None

    scan_results.append(dict(
        eta=eta,
        D0_A=float(D0_A_i),
        D0_B=float(D0_B_i),
        crossing_time=t_cross_i,
        D_A=D_A_i.tolist(),
        D_B=D_B_i.tolist(),
    ))

    ax = axes[i]
    ax.plot(times_mp, D_A_i, color=AMBER, lw=1.8)
    ax.plot(times_mp, D_B_i, color=BLUE,  lw=1.8)
    if t_cross_i:
        ax.axvline(t_cross_i, color='white', lw=0.8, ls='--', alpha=0.35)
    ax.set_title(f'η = {eta:.2f}', fontsize=9)
    ax.grid(True, alpha=0.2)
    if i == 0:
        ax.set_ylabel('D(ρ, ρ_ss)')
    ax.set_xlabel('time')

plt.tight_layout()
plt.show()

print("\nLD scan summary:")
print(f"{'η':>6}  {'D0_A':>8}  {'D0_B':>8}  {'t*':>10}")
for r in scan_results:
    tc = f"{r['crossing_time']:.2f}" if r['crossing_time'] else 'none'
    print(f"{r['eta']:>6.2f}  {r['D0_A']:>8.4f}  {r['D0_B']:>8.4f}  {tc:>10}")

---
## 7. Export to JSON

Results written to `../data/rss_dynamics.json` for website rendering.

In [ ]:
output = {
    "meta": {
        "description": "RSS finder — module 01 output",
        "system": "spin-1/2 + single motional mode + Markovian bath",
        "parameters": PARAMS_BASE,
        "units": "dimensionless (ω₀ = 1)",
        "note": "Illustrative simulation. Pre-registration pending. Not experimental data.",
        "generated": datetime.now().isoformat(),
        "tool": f"QuTiP {qt.__version__} / Python 3"
    },
    "rss_check": {
        "description": "Spin initialised in RSS. Motion thermal at n_bar. D rises then returns.",
        "times": [round(float(t), 2) for t in times_diag],
        "D_rss_init": [round(float(d), 6) for d in D_diag],
    },
    "mpemba_scenario": {
        "description": "State A (farther) vs State B (closer). Mpemba crossing if A falls below B.",
        "times": [round(float(t), 2) for t in times_mp],
        "D_state_A": [round(float(d), 6) for d in D_A],
        "D_state_B": [round(float(d), 6) for d in D_B],
        "D0_A": round(float(D0_A), 4),
        "D0_B": round(float(D0_B), 4),
        "crossing_time_computed": round(float(t_cross), 2) if t_cross else None,
    },
    "ld_scan": {
        "description": "Lamb-Dicke correction scan over η",
        "eta_values": eta_values,
        "results": [
            {
                "eta": r["eta"],
                "D0_A": round(r["D0_A"], 4),
                "D0_B": round(r["D0_B"], 4),
                "crossing_time": round(r["crossing_time"], 2) if r["crossing_time"] else None,
            }
            for r in scan_results
        ]
    }
}

out_path = os.path.join(os.path.dirname(os.getcwd()), 'data', 'rss_dynamics.json')
os.makedirs(os.path.dirname(out_path), exist_ok=True)

with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f"Exported to {out_path}")
print(f"File size: {os.path.getsize(out_path) / 1024:.1f} kB")

---
## Status

| Task | Status |
|------|--------|
| RSS solver (Markovian) | ✓ done |
| Liouvillian spectrum | ✓ done |
| RSS diagnostic check | ✓ done |
| Mpemba state pair (illustrative) | ✓ done |
| Lamb-Dicke correction scan | ✓ done |
| Platform-calibrated parameters | → Module 02 |
| Non-Markovian bath (HEOM) | → Module 04 |
| Bloch sphere scan for Mpemba manifold | → Module 03 |

---
*Pre-registration pending. 2016 dataset embargoed.*  
*quantum-relaxation-ordering · Colla, A. & Warring, U. · Freiburg · v0.1*